# MMS-300M Multilingual Evaluation (Kaggle)
Evaluates `Raemih/mms-multilingual-ser` on English, Tamil, Sinhala and saves table metrics + confusion matrices + classification reports.

In [ ]:
# =========================
# Kaggle Evaluation Script
# =========================
# Model: https://huggingface.co/Raemih/mms-multilingual-ser
# Commit reference: 8c12218e59d1e98717715b40985c2c8a581fd0bc
# Produces table metrics + confusion matrix + classification report for EN/TA/SI

!pip -q install transformers datasets torchaudio librosa soundfile scikit-learn seaborn huggingface_hub

import os
import re
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import librosa
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import Dataset, DataLoader

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Raemih/mms-multilingual-ser"
OUTPUT_DIR = Path("/kaggle/working/mms300m_eval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# From commit 8c12218...:
# phase_a_epochs = 10, phase_b_epochs = 15
EPOCHS_RUN = 25

SAMPLING_RATE = 16000
MAX_DURATION = 3.0
MAX_SAMPLES = int(SAMPLING_RATE * MAX_DURATION)
BATCH_SIZE = 8
FAST_MODE = True
MAX_SAMPLES_PER_LANG = {"english": 1000, "tamil": 500, "sinhala": 500}

# emotion order used in your multilingual script
EMOTIONS = ["neutral", "happy", "sad", "angry", "fear"]

# -------------------------
# Auto-detect datasets
# -------------------------
KAGGLE_INPUT = Path("/kaggle/input")

RAVDESS_PATH = None
TESS_PATH = None
TAMIL_PATH = None
SINHALA_PATH = None

for item in KAGGLE_INPUT.iterdir():
    name = item.name.lower()
    if "ravdess" in name:
        RAVDESS_PATH = item
    elif "tess" in name or "toronto" in name:
        TESS_PATH = item
    elif "tamil" in name or "emo" in name:
        TAMIL_PATH = item
    elif "sinhala" in name:
        SINHALA_PATH = item

print("Detected:")
print("RAVDESS:", RAVDESS_PATH)
print("TESS:", TESS_PATH)
print("TAMIL:", TAMIL_PATH)
print("SINHALA:", SINHALA_PATH)

# -------------------------
# Helpers
# -------------------------
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

def find_audio_files(base_path):
    if base_path is None:
        return []
    base_path = Path(base_path)
    if not base_path.exists():
        return []
    files = []
    for p in base_path.rglob("*"):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
            files.append(str(p))
    return files

def normalize_emotion(s: str):
    s = str(s).strip().lower()
    mapping = {
        "angry": "angry", "anger": "angry",
        "fear": "fear", "fearful": "fear",
        "happy": "happy", "happiness": "happy", "ps": "happy",
        "neutral": "neutral",
        "sad": "sad", "sadness": "sad",
    }
    return mapping.get(s, None)

def detect_emotion_from_path(path_str: str):
    t = path_str.lower()
    for k in ["neutral", "happy", "sad", "angry", "fear"]:
        if k in t:
            return k
    if "_ps" in t or "ps_" in t:
        return "happy"
    return None

def prepare_ravdess(base_path: Path):
    rows = []
    emo_map = {"01":"neutral","03":"happy","04":"sad","05":"angry","06":"fear"}
    for fp in find_audio_files(base_path):
        fn = Path(fp).stem
        parts = fn.split("-")
        if len(parts) >= 3:
            code = parts[2]
            if code in emo_map:
                rows.append({"path": fp, "emotion": emo_map[code], "language": "english"})
    return pd.DataFrame(rows)

def prepare_tess(base_path: Path):
    rows = []
    for fp in find_audio_files(base_path):
        emo = detect_emotion_from_path(fp)
        if emo in EMOTIONS:
            rows.append({"path": fp, "emotion": emo, "language": "english"})
    return pd.DataFrame(rows)

def prepare_tamil(base_path: Path):
    rows = []
    if base_path is None:
        return pd.DataFrame(columns=["path","emotion","language"])
    for fp in find_audio_files(base_path):
        emo = detect_emotion_from_path(fp)
        if emo is None:
            stem = Path(fp).stem.lower()
            for e in EMOTIONS:
                if stem.endswith(f"_{e}") or f"_{e}_" in stem:
                    emo = e
                    break
        if emo in EMOTIONS:
            rows.append({"path": fp, "emotion": emo, "language": "tamil"})
    return pd.DataFrame(rows)

def prepare_sinhala(base_path):
    rows = []
    if base_path is None:
        return pd.DataFrame(columns=["path","emotion","language"])
    base_path = Path(base_path)
    if not base_path.exists():
        return pd.DataFrame(columns=["path","emotion","language"])

    csv_candidates = list(base_path.rglob("*.csv"))
    loaded_from_csv = False

    for csv_fp in csv_candidates:
        try:
            df = pd.read_csv(csv_fp)
        except Exception:
            continue

        cols = {c.lower(): c for c in df.columns}
        path_col = None
        emo_col = None

        for c in ["path","filepath","file_path","audio","audio_path","filename","file"]:
            if c in cols:
                path_col = cols[c]
                break
        for c in ["emotion","label","class","target"]:
            if c in cols:
                emo_col = cols[c]
                break

        if path_col and emo_col:
            for _, r in df.iterrows():
                relp = str(r[path_col])
                emo = normalize_emotion(r[emo_col])
                if emo not in EMOTIONS:
                    emo = detect_emotion_from_path(relp)
                if emo not in EMOTIONS:
                    continue
                fullp = Path(relp)
                if not fullp.is_absolute():
                    fullp = (csv_fp.parent / relp).resolve()
                if fullp.exists() and fullp.suffix.lower() in AUDIO_EXTS:
                    rows.append({"path": str(fullp), "emotion": emo, "language": "sinhala"})
            if len(rows) > 0:
                loaded_from_csv = True
                break

    if not loaded_from_csv:
        for fp in find_audio_files(base_path):
            emo = detect_emotion_from_path(fp)
            if emo in EMOTIONS:
                rows.append({"path": fp, "emotion": emo, "language": "sinhala"})

    return pd.DataFrame(rows)

def stratified_split(df, test_size=0.15, val_size=0.15, seed=42):
    if len(df) < 20:
        train_df = df.sample(frac=0.7, random_state=seed)
        rem = df.drop(train_df.index)
        val_df = rem.sample(frac=0.5, random_state=seed)
        test_df = rem.drop(val_df.index)
        return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

    train_df, temp_df = train_test_split(
        df, test_size=(test_size + val_size), random_state=seed, stratify=df["emotion"]
    )
    val_ratio_of_temp = val_size / (test_size + val_size)
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - val_ratio_of_temp), random_state=seed, stratify=temp_df["emotion"]
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

# -------------------------
# Build datasets
# -------------------------
ravdess_df = prepare_ravdess(RAVDESS_PATH) if RAVDESS_PATH else pd.DataFrame(columns=["path","emotion","language"])
tess_df = prepare_tess(TESS_PATH) if TESS_PATH else pd.DataFrame(columns=["path","emotion","language"])
english_df = pd.concat([ravdess_df, tess_df], ignore_index=True)

tamil_df = prepare_tamil(TAMIL_PATH)
sinhala_df = prepare_sinhala(SINHALA_PATH)

for name, df in [("english", english_df), ("tamil", tamil_df), ("sinhala", sinhala_df)]:
    if len(df) == 0:
        raise RuntimeError(f"No samples found for {name}. Check Kaggle dataset paths/labels.")
    df = df[df["emotion"].isin(EMOTIONS)].copy()
    if name == "english":
        english_df = df
    elif name == "tamil":
        tamil_df = df
    else:
        sinhala_df = df


if FAST_MODE:
    def _cap(df, n):
        if len(df) <= n:
            return df.reset_index(drop=True)
        return df.sample(n=n, random_state=SEED).reset_index(drop=True)

    english_df = _cap(english_df, MAX_SAMPLES_PER_LANG["english"])
    tamil_df = _cap(tamil_df, MAX_SAMPLES_PER_LANG["tamil"])
    sinhala_df = _cap(sinhala_df, MAX_SAMPLES_PER_LANG["sinhala"])
    print("\nFAST_MODE enabled: dataset capped for runtime")

print("\nSample counts:")
print("English:", len(english_df))
print("Tamil:", len(tamil_df))
print("Sinhala:", len(sinhala_df))

splits = {}
for lang, df in [("english", english_df), ("tamil", tamil_df), ("sinhala", sinhala_df)]:
    tr, va, te = stratified_split(df, test_size=0.15, val_size=0.15, seed=SEED)
    splits[lang] = {"train": tr, "val": va, "test": te}
    print(f"{lang:8s} -> train={len(tr)}, val={len(va)}, test={len(te)}")

# -------------------------
# Load model + processor
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

# Robust loader: supports repos missing `model_type` in config.json
model_id2label = {}
try:
    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID, trust_remote_code=True)
except Exception:
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)

try:
    model = AutoModelForAudioClassification.from_pretrained(MODEL_ID, trust_remote_code=True).to(device)
except Exception as e:
    print("Auto loader failed, falling back to Wav2Vec2ForSequenceClassification:", e)
    model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_ID).to(device)

model.eval()

cfg = getattr(model, "config", None)
if cfg is not None and hasattr(cfg, "id2label") and cfg.id2label:
    for k, v in cfg.id2label.items():
        try:
            model_id2label[int(k)] = str(v).lower()
        except Exception:
            try:
                model_id2label[int(str(k))] = str(v).lower()
            except Exception:
                pass

if model_id2label:
    emotion_to_model_id = {}
    for e in EMOTIONS:
        found = None
        for i, n in model_id2label.items():
            if e in n:
                found = i
                break
        if found is None:
            found = EMOTIONS.index(e)
        emotion_to_model_id[e] = found
else:
    emotion_to_model_id = {e: i for i, e in enumerate(EMOTIONS)}

print("Emotion->Model ID mapping:", emotion_to_model_id)

# -------------------------
# Dataset class
# -------------------------
class AudioEmotionDataset(Dataset):
    def __init__(self, df, feature_extractor, emotion_to_id):
        self.df = df.reset_index(drop=True)
        self.fx = feature_extractor
        self.emotion_to_id = emotion_to_id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = librosa.load(row["path"], sr=SAMPLING_RATE)
        if len(audio) > MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        inputs = self.fx(
            audio,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SAMPLES
        )
        label_id = self.emotion_to_id[row["emotion"]]
        item = {
            "input_values": inputs["input_values"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0) if "attention_mask" in inputs else None,
            "labels": torch.tensor(label_id, dtype=torch.long),
            "emotion_str": row["emotion"],
            "path": row["path"]
        }
        return item

def collate_fn(batch):
    input_values = [b["input_values"] for b in batch]
    labels = torch.stack([b["labels"] for b in batch])

    lengths = [x.shape[-1] for x in input_values]
    max_len = max(lengths)
    padded = []
    attn = []
    for x in input_values:
        pad_len = max_len - x.shape[-1]
        if pad_len > 0:
            x = torch.nn.functional.pad(x, (0, pad_len))
        padded.append(x)
        mask = torch.ones(max_len, dtype=torch.long)
        if pad_len > 0:
            mask[-pad_len:] = 0
        attn.append(mask)

    return {
        "input_values": torch.stack(padded),
        "attention_mask": torch.stack(attn),
        "labels": labels
    }

def extract_emotion_logits(outputs):
    if hasattr(outputs, "emotion_logits"):
        return outputs.emotion_logits
    if hasattr(outputs, "logits"):
        logits = outputs.logits
        if isinstance(logits, (tuple, list)):
            return logits[0]
        return logits
    if hasattr(outputs, "classifier_logits"):
        return outputs.classifier_logits
    if isinstance(outputs, dict):
        if "emotion_logits" in outputs:
            return outputs["emotion_logits"]
        if "logits" in outputs:
            lg = outputs["logits"]
            if isinstance(lg, (tuple, list)):
                return lg[0]
            return lg
    if isinstance(outputs, (tuple, list)):
        return outputs[0]
    raise RuntimeError("Could not locate emotion logits in model output.")

def evaluate_split(model, loader):
    all_preds, all_labels = [], []
    total_loss = 0.0
    n_batches = 0
    ce = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in loader:
            input_values = batch["input_values"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_values=input_values, attention_mask=attention_mask)
            logits = extract_emotion_logits(outputs)

            loss = ce(logits, labels)
            total_loss += loss.item()
            n_batches += 1

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, n_batches)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)

def save_reports(lang, split_name, y_true, y_pred, id_to_emotion):
    lang_dir = OUTPUT_DIR / lang / split_name
    lang_dir.mkdir(parents=True, exist_ok=True)

    target_names = [id_to_emotion[i] for i in sorted(id_to_emotion.keys())]
    labels_sorted = sorted(id_to_emotion.keys())

    cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)
    cr = classification_report(
        y_true, y_pred, labels=labels_sorted, target_names=target_names, digits=4, output_dict=True
    )

    with open(lang_dir / "classification_report.json", "w") as f:
        json.dump(cr, f, indent=2)

    with open(lang_dir / "classification_report.txt", "w") as f:
        f.write(classification_report(y_true, y_pred, labels=labels_sorted, target_names=target_names, digits=4))

    np.save(lang_dir / "confusion_matrix.npy", cm)

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f"Confusion Matrix - {lang} - {split_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(lang_dir / "confusion_matrix.png", dpi=200)
    plt.close()

# -------------------------
# Evaluate all languages
# -------------------------
id_to_emotion = {v: k for k, v in emotion_to_model_id.items()}

rows = []
for lang in ["english", "tamil", "sinhala"]:
    print(f"\nEvaluating {lang.upper()}...")

    ds_train = AudioEmotionDataset(splits[lang]["train"], feature_extractor, emotion_to_model_id)
    ds_val   = AudioEmotionDataset(splits[lang]["val"], feature_extractor, emotion_to_model_id)
    ds_test  = AudioEmotionDataset(splits[lang]["test"], feature_extractor, emotion_to_model_id)

    dl_val   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
    dl_test  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

    if FAST_MODE:
        tr_loss, tr_acc = None, None
        ytr, ptr = None, None
    else:
        dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
        tr_loss, tr_acc, ytr, ptr = evaluate_split(model, dl_train)
        save_reports(lang, "train", ytr, ptr, id_to_emotion)

    va_loss, va_acc, yva, pva = evaluate_split(model, dl_val)
    te_loss, te_acc, yte, pte = evaluate_split(model, dl_test)

    save_reports(lang, "val", yva, pva, id_to_emotion)
    save_reports(lang, "test", yte, pte, id_to_emotion)

    rows.append({
        "language": lang,
        "train_accuracy": tr_acc,
        "train_loss": tr_loss,
        "validation_accuracy": va_acc,
        "validation_loss": va_loss,
        "test_accuracy": te_acc,
        "test_loss": te_loss,
        "epochs_run": EPOCHS_RUN
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_DIR / "mms300m_metrics_table.csv", index=False)

print("\n=== FINAL TABLE ===")
if FAST_MODE:
    print("FAST_MODE: train metrics skipped (set to None) to stay under runtime target")
display(summary_df)

md_cols = ["language","train_accuracy","train_loss","validation_accuracy","validation_loss","epochs_run","test_accuracy","test_loss"]
print("\nMarkdown:")
print("| " + " | ".join(md_cols) + " |")
print("|" + "|".join(["---"]*len(md_cols)) + "|")
for _, r in summary_df.iterrows():
    print(
        f"| {r['language']} | "
        f"{r['train_accuracy']:.4f} | {r['train_loss']:.4f} | "
        f"{r['validation_accuracy']:.4f} | {r['validation_loss']:.4f} | "
        f"{int(r['epochs_run'])} | "
        f"{r['test_accuracy']:.4f} | {r['test_loss']:.4f} |"
    )

print(f"\nSaved to: {OUTPUT_DIR}")
